In [1]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess
from string import Template
from IPython.display import Markdown, display


with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils
import llm_tools as llt

from matplotlib import pyplot as plt


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [2]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 8000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 4000
REQUESTS_PER_BATCH = 1000
REPLACE = False

In [3]:
rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)


In [4]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [5]:
# Summarize the differences between the clusters

PROMPT = Template("""
In what follows, I will provide you with summaries of agenda items from Los Angeles City Planning Commission meetings. The agenda items have been grouped into three clusters. Several examples are provided from each cluster. Summarize the typical kind of case each cluster contains, and highlight the main differences between the clusters.
                  
$examples
""")

examples_text = ""
for cluster in [0, 1, 2]:
    mask = df['cluster']==cluster
    rows = df.loc[mask].sample(7, random_state=rng).reset_index(drop=True)
    examples_text += "------------------------\n"
    examples_text += f"Cluster: {cluster}\n\n"
    for i, row in rows.iterrows():
        examples_text += f"Example {i+1}:\n"
        examples_text += f"{row['agenda_summary']}\n\n"

prompt = PROMPT.substitute(examples=examples_text)

response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)

display(Markdown(response['response']))


## Cluster Summaries and Comparison

### Cluster 0: Appeals of Subdivision/Tract Map Approvals
These cases are appeals of the Advisory Agency's approval of **Vesting Tentative Tract Maps** for large-scale development projects. They typically involve the **merger and re-subdivision of land** into ground lots and airspace lots (often for condominium purposes), **haul routes** for the export of large quantities of soil, and associated **environmental review** (EIR certification, Mitigated Negative Declarations, or SCEAs). The underlying projects tend to be substantial — large mixed-use developments, studio campuses, entertainment complexes, or significant residential subdivisions — but the procedural focus at the Commission level is on the tract map and the adequacy of the environmental determination. Projects frequently involve tens of thousands to hundreds of thousands of cubic yards of soil export and may include street vacations, alley mergers, or dedications.

### Cluster 1: Miscellaneous Entitlements, Zone Changes, Specific Plans, and Procedural Items
This cluster is the most heterogeneous. It encompasses a wide range of planning actions that do not fit neatly into the subdivision or density bonus/affordable housing categories. Cases include **zone changes** (city-initiated or project-specific), **Specific Plan establishments**, **Conditional Use Permits** (e.g., for schools in residential zones), **Plan Approvals** evaluating compliance with prior conditions, **property sales/exchanges**, **modifications to recorded tract map conditions**, and even **withdrawn items**. The projects vary enormously in scale and type — from a private school reuse to a 25-acre studio modernization to a city-initiated rezoning of 23 lots. The common thread is that these are discretionary planning actions or policy-level decisions that go beyond standard development entitlements.

### Cluster 2: Density Bonus / TOC Affordable Housing Residential Projects (Often Appeals)
These cases center on **multi-family residential (or mixed-use) developments** that utilize the **Density Bonus Law** or the **Transit Oriented Communities (TOC) Affordable Housing Incentive Program** to achieve greater density, height, and FAR in exchange for reserving units for Very Low Income or Extremely Low Income households. They are frequently **appeals of the Director of Planning's approval** of these incentive-based entitlements. The projects are typically mid-rise buildings (5–8 stories) on relatively modest urban lots, and the incentives at issue commonly include increased density, increased FAR, reduced parking requirements, reduced setbacks, reduced open space, and additional height. Environmental review is usually a categorical exemption or Mitigated Negative Declaration rather than a full EIR.

---

## Key Differences

| Dimension | Cluster 0 | Cluster 1 | Cluster 2 |
|---|---|---|---|
| **Core action** | Vesting Tentative Tract Map approval (subdivision) | Zone changes, Specific Plans, CUPs, plan approvals, and other miscellaneous entitlements | Density Bonus / TOC incentive approvals |
| **Appealed from** | Advisory Agency | Varies (Director, City-initiated, or original jurisdiction) | Director of Planning |
| **Project scale** | Large (multi-acre sites, major campuses, large mixed-use) | Highly variable | Moderate (single-parcel infill, mid-rise buildings) |
| **Affordable housing focus** | Not central | Generally not central | Central — affordability covenants are the mechanism enabling the incentives |
| **Environmental review** | Often involves EIR certification or substantial CEQA documents | Varies widely | Typically categorical exemptions or MNDs |
| **Recurring technical issues** | Lot mergers/resubdivisions, haul routes, soil export, street dedications/vacations | Zoning/land use policy, conditional uses, compliance monitoring | FAR increases, height bonuses, parking reductions, setback reductions, open space reductions |
| **Homogeneity** | Fairly uniform procedural pattern | Most diverse cluster | Most uniform — nearly all follow the same density bonus/TOC template |

In essence, **Cluster 0** deals with the **land subdivision mechanics** of large projects, **Cluster 1** is a **catch-all** for diverse planning entitlements and policy actions, and **Cluster 2** focuses on **incentive-based affordable housing developments** seeking density and dimensional relief through the Density Bonus or TOC programs.

In [6]:
text = r"""Cluster 0, the smallest cluster, consists mainly of non-development actions, such as citywide code amendments, plan updates, or conditional use permits on existing properties. 
Cluster 1 consists primarily of large 
scale private development projects in which the CPC is the initial approver. 
Cluster 2 consists primarily of appeals of decisions made by lower level agencies."""

with open(os.path.join(LOCAL_PATH, "results/cluster_summaries.tex"), 'w') as f:
    f.write(text)

In [7]:
# Summarize the differences between the clusters

PROMPT = Template("""
In what follows, I will provide you with summaries of agenda items from Los Angeles City Planning Commission meetings. The first example has been identified via semantic embeddings as being especially unique or an outlier relative to the other examples. Please describe what makes this example unique compared to the others.
                  
$examples
""")

output_text = ""
for cluster in [0, 1, 2]:
    output_text += "------------------------\n"
    output_text += f"# Cluster: {cluster}\n\n"
    mask = df['cluster']==cluster
    regular_examples = df.loc[mask].sample(7, random_state=rng).reset_index(drop=True)
    mask2 = (df['cluster']==cluster) 
    unique_example = df.loc[mask2].sort_values(by='atypicality', ascending=False).head(1).reset_index(drop=True).iloc[0]
    examples_text = f"## Unique Example:\n\n"
    examples_text += f"{unique_example['agenda_summary']}\n\n"
    output_text += f"## Unique Example:\n\n"
    output_text += f"{unique_example['date']} - {unique_example['item_no']}\n\n" 
    output_text += f"{unique_example['agenda_summary']}\n\n"
    for i, row in regular_examples.iterrows():
        examples_text += f"## Regular Example {i+1}:\n\n"
        examples_text += f"{row['agenda_summary']}\n\n"
    prompt = PROMPT.substitute(examples=examples_text)
    response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    output_text += "# LLM Response:\n\n"
    output_text += f"{response['response']}\n\n"

display(Markdown(output_text))


------------------------
# Cluster: 0

## Unique Example:

2018-12-13 - 7

Appeal of the Deputy Advisory Agency's approval of a Vesting Tentative Tract Map (VTT-74200) on a 4.9-acre site at 129-135 West College Street and 924 North Spring Street in the Central City North Plan Area. The project involves street vacation purposes. The appeal challenges both the certification of the College Station Project EIR and the approval of the Vesting Tentative Tract Map. The applicant is Jeffrey Goldberger of Atlas Capital Group LLC, and the appellants are labor and community organizations.

# LLM Response:

## What Makes the Unique Example Distinctive

The unique example stands out from the regular examples in several key ways:

### 1. **Street Vacation as the Primary Purpose**
The most striking difference is that the Vesting Tentative Tract Map is for **"street vacation purposes."** This is fundamentally different from every regular example, where tract maps serve purposes like:
- Condominium subdivision
- Small lot subdivision
- Merger and re-subdivision into ground lots and airspace lots
- Haul route approvals

Street vacation — the process of converting public street right-of-way into private property — is a distinct land use action not seen in any of the other examples.

### 2. **Absence of Specific Development Details**
Every regular example includes detailed descriptions of the proposed development: unit counts, square footages, building heights, parking spaces, income-restricted units, commercial space, etc. The unique example provides **none of this**. It references the "College Station Project EIR," suggesting a larger development context, but the agenda item itself is framed around the street vacation rather than a specific building program. This makes it unusually abstract compared to the others.

### 3. **Appellants Are Labor and Community Organizations (Plural, Unnamed)**
While several regular examples feature labor-affiliated appellants (CREED LA, Southwest Regional Council of Carpenters, Laborer's International Union Local 300), they are specifically named. The unique example refers generically to **"labor and community organizations"** — a vague, coalition-style description that is less precise and suggests a broader opposition front.

### 4. **Named Project (College Station Project)**
The item references a named project with its own EIR, suggesting it is part of a **larger, well-known development** with a distinct identity. While Regular Example 6 mentions the "Sunset Wilcox Project," most other examples are identified only by address and tract number.

### Summary
The core distinguishing feature is the **street vacation purpose** — a qualitatively different type of land use action compared to the residential/commercial subdivision and development projects that characterize all the regular examples. Combined with the lack of development specifics and the generic identification of appellants, this item reads as procedurally and substantively distinct from the typical tract map appeal.

------------------------
# Cluster: 1

## Unique Example:

2023-08-24 - 4

This item concerns action consistent with the terms of a court-issued Writ in Hi Point Neighbor's Association v. City of Los Angeles. The Revised Peremptory Writ of Mandamus, issued May 25, 2023, orders the City to set aside its Tier 3 TOC Affordable Housing Incentive Program approval (DIR-2020-2067-TOC) for a proposed 20-unit multi-family residential building at 1447 South Hi Point Street in the Wilshire Plan Area. The Writ does not limit or control the discretion vested in the City.

# LLM Response:

## What Makes the Unique Example Distinct

The unique example stands apart from all the regular examples in several fundamental ways:

### 1. **It's a Legal/Judicial Compliance Action, Not a Development Proposal**
Every regular example involves a proactive land use application — someone proposing to build, renovate, convert, or expand a physical project (a hotel, storage facility, hospital wing, school relocation, research building, signage agreement, or auto service center). The unique example involves no new construction or development whatsoever. Instead, it is a **response to a court order** (a Revised Peremptory Writ of Mandamus), requiring the City to **undo** a previous approval. The Commission is being asked to comply with a judicial mandate, not exercise its typical planning discretion over a new project.

### 2. **Retroactive/Remedial Rather Than Forward-Looking**
The regular examples all look forward — they seek new entitlements, zone changes, conditional uses, environmental clearances, and site plan reviews for proposed projects. The unique example looks **backward**, requiring the City to **set aside** an approval it had already granted (DIR-2020-2067-TOC). It is fundamentally a reversal rather than an authorization.

### 3. **Absence of Typical Planning Entitlement Requests**
The regular examples are characterized by detailed lists of requested actions: Mitigated Negative Declarations, Zone Changes, Conditional Uses, Variances, Site Plan Reviews, Development Agreements, CEQA clearances, Specific Plan Amendments, etc. The unique example contains **none of these standard entitlement mechanisms**. Its procedural framework is entirely different — rooted in litigation and court orders rather than the municipal planning code.

### 4. **Driven by an External Party (the Court) Rather Than an Applicant**
In every regular example, the action is initiated by a developer, institution, or the Department of City Planning itself. The unique example is driven by a **neighborhood association's successful lawsuit** against the City, with the court compelling the Commission's action. The Commission's role here is not to evaluate a proposal on its merits but to comply with a judicial directive.

### 5. **Involves the TOC Incentive Program Specifically**
While the regular examples involve conventional zoning tools, the unique example concerns the **Transit Oriented Communities (TOC) Affordable Housing Incentive Program** — a policy mechanism that grants density bonuses and development incentives near transit. The legal challenge to this program-level approval gives the item a policy/programmatic dimension absent from the site-specific development reviews in the other examples.

### 6. **Scale and Simplicity**
The project at issue (a 20-unit residential building) is notably modest compared to the large-scale developments in the regular examples. But more importantly, the item's substance is not about the building's details at all — it's about the **legal process** surrounding its approval.

In summary, this item is unique because it represents the Planning Commission acting not in its typical quasi-legislative or quasi-judicial planning role, but as an **instrument of court-ordered compliance**, reversing a prior decision as a result of successful litigation.

------------------------
# Cluster: 2

## Unique Example:

2025-11-06 - 10

This item involves an appeal of a Department of Building and Safety action to issue a demolition permit for a one-story duplex located at 2656 (2654) South Magnolia Avenue in South Los Angeles. The Associate Zoning Administrator previously upheld LADBS's issuance of the demolition permit. The West Adams Heritage Association, along with other appellants, is now appealing that determination to the Planning Commission.

# LLM Response:

## What Makes the Unique Example an Outlier

The unique example stands out from the regular examples in several fundamental ways:

### 1. **It concerns only a demolition permit — with no new construction proposed**
Every regular example involves a development project where demolition (if any) is merely a precursor to building something new — apartment buildings, mixed-use developments, medical offices, eldercare facilities, or condominiums. The unique example is solely about whether a demolition permit for an existing duplex should have been issued. There is no proposed replacement structure, no new units, no development plan attached.

### 2. **It challenges a Department of Building and Safety (LADBS) administrative action, not a planning entitlement**
The regular examples all involve planning-level approvals: Density Bonus Compliance Reviews, TOC incentive programs, Zone Changes, Site Plan Reviews, Coastal Development Permits, Specific Plan Compliance, CEQA determinations, and similar land-use entitlements. The unique example instead involves an appeal of a ministerial building permit action — the issuance of a demolition permit by LADBS — which was first reviewed by a Zoning Administrator before reaching the Commission. This is a fundamentally different procedural pathway.

### 3. **The appellant is a heritage/preservation organization, not a neighbor or development opponent**
The West Adams Heritage Association is appealing on what appear to be historic preservation grounds, seeking to prevent demolition of an existing structure. In the regular examples, appellants (when identified) tend to be neighbors, community groups like SAFER, or individuals challenging the scope or impacts of new development.

### 4. **No CEQA review, density bonuses, affordable housing components, or land-use incentives are involved**
The regular examples are dense with planning mechanisms — TOC tiers, Very Low/Extremely Low Income unit set-asides, FAR and height incentives, Mitigated Negative Declarations, zone changes, and specific plan compliance. The unique example involves none of these. It is procedurally and substantively minimalist by comparison.

### 5. **Scale and complexity**
The regular examples involve multi-story, multi-unit projects on the order of 8 to 234 units, with complex entitlement packages often requiring multiple discretionary approvals. The unique example concerns a single one-story duplex — a small-scale structure — and a single procedural question: was the demolition permit properly issued?

In essence, this item is an outlier because it is a **preservation-motivated challenge to a simple demolition permit** rather than a review or appeal of a **new development project with associated planning entitlements**.



In [8]:
text = r"""In cluster 0, the most atypical case was an application for a conditional use permit to operate cultural events such as charitable fundraisers, film screenings, and concerts, and to sell alcoholic beverages on an existing 53-acre cemetery. 
In cluster 1, the most atypical case was an application to permit development of a 143.5 foot tall, 201,292 square foot multi-disciplinary research facility on the USC Health Sciences Campus.
In cluster 2, the most atypical case involved a Writ of Mandamus issued by the Los Angeles Superior Court ordering the City to set aside a previous approval on a 20-unit housing development because it hadn't adequately demonstrated TOC Tier 3 eligibility. However, the Writ did not limit the discretion vested in the City as to how to proceed with the project."""

with open(os.path.join(LOCAL_PATH, "results/unique_examples.tex"), 'w') as f:
    f.write(text)
